In [3]:
import os
import re
import requests
import pandas as pd
from dotenv import load_dotenv
from time import sleep

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7) if os.getenv(f"GITHUB_TOKEN_{i}")]
token_index = 0
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")

def get_auth_header():
    global token_index
    header = {"Authorization": f"token {tokens[token_index]}"}
    token_index = (token_index + 1) % len(tokens)
    return header

# === Output directory and input path ===
OUTPUT_DIR = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline"
os.makedirs(OUTPUT_DIR, exist_ok=True)

input_path = os.path.join(OUTPUT_DIR, "step3_removal_keyword_output.csv")
output_path = os.path.join(OUTPUT_DIR, "step4_ci_detection_output.csv")

# === Combined CI file patterns ===
ci_patterns = {
    r'\.travis\.yml$': 'Travis CI',
    r'\.appveyor\.yml$': 'AppVeyor',
    r'appveyor\.yml$': 'AppVeyor',
    r'circle\.yml$': 'CircleCI',
    r'\.circleci/config\.yml$': 'CircleCI',
    r'azure-pipelines\.yml$': 'Azure Pipelines',
    r'\.github/workflows/.*\.(yml|yaml)$': 'GitHub Actions',
    r'bitbucket-pipelines\.yml$': 'Bitbucket',
    r'\.gitlab-ci\.yml$': 'GitLab',
    r'Jenkinsfile\.yml$': 'Jenkins',
    r'bitrise\.yml$': 'Bitrise',
    r'bamboo\.yml$': 'Bamboo',
    r'codeship-services\.yml$': 'Codeship',
    r'\.gocd\.yaml$': 'GoCD',
    r'\.cirrus\.yml$': 'Cirrus',
    r'wercker\.yaml$': 'Wercker',
    r'semaphore\.yml$': 'Semaphore',
    r'codemagic\.yaml$': 'Nevercode',
}

# === Load data ===
if os.path.exists(output_path):
    df = pd.read_csv(output_path)
else:
    df = pd.read_csv(input_path)
    df["Repository"] = df["html_url"].apply(lambda url: '/'.join(url.strip('/').split('/')[-2:]))
    df["nbr_of_yml"] = df.get("nbr_of_yml", 0)
    df["yml_detected"] = df.get("yml_detected", "none")
    df["CI_Service"] = df.get("CI_Service", "none")
    df["Valid_Repo_Step4"] = df.get("Valid_Repo_Step4", "none")

# === Progression setup ===
repos_to_review = df[(df["Valid_Repo_Step3"] == "yes") & (~df["Valid_Repo_Step4"].isin(["yes", "no"]))]
total = len(repos_to_review)
counter = 1

# === Detection loop ===
for idx, row in df.iterrows():
    if row["Valid_Repo_Step3"] != "yes" or row["Valid_Repo_Step4"] in ["yes", "no"]:
        continue

    repo = row["Repository"]
    print(f"[{counter}/{total}] Processing {repo}...")
    counter += 1

    try:
        url = f"https://api.github.com/repos/{repo}/git/trees/HEAD?recursive=1"
        res = requests.get(url, headers=get_auth_header())
        if res.status_code != 200:
            print(f"⚠️ Skipping due to fetch error: {res.status_code}")
            continue

        tree = res.json().get("tree", [])
        yml_files = [item["path"] for item in tree if item["path"].endswith((".yml", ".yaml"))]
        df.at[idx, "nbr_of_yml"] = len(yml_files)

        detected_services = set()
        for yml in yml_files:
            for pattern, service in ci_patterns.items():
                if re.search(pattern, yml):
                    detected_services.add(service)

        if detected_services:
            df.at[idx, "yml_detected"] = "yes"
            df.at[idx, "CI_Service"] = ", ".join(sorted(detected_services))
            df.at[idx, "Valid_Repo_Step4"] = "yes"
        else:
            df.at[idx, "yml_detected"] = "no"
            df.at[idx, "CI_Service"] = "none"
            df.at[idx, "Valid_Repo_Step4"] = "no"

    except Exception as e:
        print(f"❌ Error with {repo}: {e}")
        df.at[idx, "Valid_Repo_Step4"] = "error"
        df.at[idx, "yml_detected"] = "error"
        df.at[idx, "CI_Service"] = "error"

    df.to_csv(output_path, index=False)

# === Cleanup and save final output ===
if "Repository" in df.columns:
    df.drop(columns=["Repository"], inplace=True)
df.to_csv(output_path, index=False)

print(f"✅ Step 4 complete. Output saved to: {output_path}")


[1/14690] Processing Dawnthorn/nagare...
[2/14690] Processing bpellin/keepassdroid...
[3/14690] Processing connectbot/connectbot...
[4/14690] Processing JakeWharton/SMSMorse...
[5/14690] Processing JakeWharton/SMSBarrage...
[6/14690] Processing millenomi/diceshaker...
[7/14690] Processing pocmo/Yaaic...
[8/14690] Processing konklone/congress-android...
[9/14690] Processing johannilsson/sthlmtraveling...
[10/14690] Processing yaxim-org/yaxim...
[11/14690] Processing talklittle/reddit-is-fun...
[12/14690] Processing ProjectCCNx/ccnx...
[13/14690] Processing konklone/campyre...
[14/14690] Processing davidw/hecl...
[15/14690] Processing XCSoar/XCSoar...
[16/14690] Processing ligi/gobandroid...
[17/14690] Processing commonsguy/cw-android...
[18/14690] Processing matburt/mobileorg-android...
[19/14690] Processing ChainsDD/Superuser...
[20/14690] Processing jwood/standup-timer...
[21/14690] Processing chrisboyle/sgtpuzzles...
[22/14690] Processing quran/quran_android...
[23/14690] Processing 